# Arabic Cursive OCR Benchmark — Coordinated Local Sweep (5 core models)

Same coordinated multi-runner **full sweep** as `04_colab_sweep.ipynb`, but
restricted to the five local models already on the leaderboard
(`qari-ocr-0.2`, `qari-ocr-0.3`, `qwen2-vl-2b`, `qwen2.5-vl-3b`,
`trocr-large-arabic`) — the heavier / newer models (`qwen2.5-vl-7b`,
`qwen3-vl-8b`, the `nakba-*` systems, `ain-7b`, `minicpm-v`) stay in
notebook 04. Every `(model, dataset)` pair runs to **full split coverage**.

It used to stop at 20 lines/pair; now `TARGET = None` (full). Set `TARGET`
to an int if you want a quick capped pass instead.

**Multi-runner + resumable:** identical claims-ledger coordination and
per-line cache as notebook 04 (see the Notes at the bottom). One runner
alone works too.

**Before running:** `Runtime -> T4 GPU`; add `GH_TOKEN` (write-scoped) and
`HF_TOKEN` secrets (see `notebooks/README.md`).

## 1. GPU (falls back to CPU)

In [ ]:
import torch
HAS_GPU = torch.cuda.is_available()
if HAS_GPU:
    print('GPU:', torch.cuda.get_device_name(0))
    !nvidia-smi --query-gpu=memory.total,memory.used --format=csv
else:
    print('NO GPU — will run on CPU (much slower; big 4-bit models are skipped).')
    print('For a real run: Runtime -> Change runtime type -> T4 GPU.')

## 2. Repo

In [ ]:
# Bulletproof checkout: works whether the repo is missing, at the wrong
# path, or has local edits. Always ends up at the latest origin/main.
import os
REPO = '/content/arabic-ocr-benchmark'
# Copy Colab secrets into the environment. SWEEP_RUNNER_ID pins a STABLE
# name for this account (e.g. 'runner-a') so a restarted runner re-adopts
# its own in-flight batches instantly instead of waiting out the lease TTL.
for _k in ('GH_TOKEN', 'HF_TOKEN', 'SWEEP_RUNNER_ID'):
    try:
        from google.colab import userdata
        _v = userdata.get(_k)
        if _v: os.environ[_k] = _v
    except Exception:
        pass
token = os.environ.get('GH_TOKEN')
if not token:
    print('no GH_TOKEN secret — clone/push will be anonymous / read-only')
url = f"https://{(token + '@') if token else ''}github.com/rabeeh97/arabic-ocr-benchmark.git"
if os.path.isdir(REPO + '/.git'):
    os.system(f'git -C {REPO} fetch --depth 1 origin main && git -C {REPO} reset --hard origin/main')
    # drop untracked leftovers (e.g. a stale sweep_manifest.json) so the
    # manifest is always exactly what's in the repo. Keeps gitignored _cache.
    os.system(f'git -C {REPO} clean -fd -- results')
else:
    os.system(f'rm -rf {REPO} && git clone --depth 1 {url} {REPO}')
os.chdir(REPO)
# src-layout package: importable straight from the checkout even if the
# `pip install -e` cell fails or is skipped (this is what caused the old
# 'No module named arabic_ocr_bench' in the sweep cell).
import sys
for _p in (os.path.join(os.getcwd(), 'src'), os.getcwd()):
    if _p not in sys.path: sys.path.insert(0, _p)
print(os.popen('git log --oneline -3').read())

## 3. Install + caches

In [ ]:
# Keep all downloads on Colab's own scratch disk (ephemeral, ~78 GB) --
# never the local machine. Nothing here touches your computer.
import os
os.environ['HF_HOME'] = '/content/hf_cache'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
# Optional HF_TOKEN secret: lifts anonymous rate limits and speeds the
# Muharaf download (and is required for gated models like PaliGemma's base).
try:
    from google.colab import userdata
    _hf = userdata.get('HF_TOKEN')
    if _hf:
        os.environ['HF_TOKEN'] = _hf
        print('HF_TOKEN set')
except Exception:
    pass

In [ ]:
!pip install -q -e '.[local]'
!pip uninstall -y -q torchao 2>/dev/null; echo 'torchao removed (unused)'

## 4. Mount Drive (shared across all runners)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 5. Config

`TARGET = None` sweeps every line of each split; set it to an int to cap
each `(model, dataset)` pair. `BATCH` is how many rows a runner claims at
once. `LEASE_TTL_MIN` is how long a claim is honoured before other runners
may reclaim it.

In [ ]:
MODELS = [
    'qari-ocr-0.2', 'qari-ocr-0.3', 'qwen2-vl-2b', 'qwen2.5-vl-3b',
    'trocr-large-arabic',
]
DATASETS = [
    'kitab_khatt', 'kitab_muharaf', 'kitab_historicalbooks', 'kitab_hindawi',
    'kitab_historyar', 'kitab_onlinekhatt', 'muharaf_lines', 'nakba_lines',
]
TARGET = None            # None = full split (every line). Set an int to cap per (model, dataset).
BATCH = 20               # rows claimed + scored per batch
LEASE_TTL_MIN = 30       # a claim older than this is reclaimable by another runner

In [ ]:
import yaml
_mc = yaml.safe_load(open('configs/models.yaml'))
if not HAS_GPU:
    # CPU: drop 4-bit models (GPU-only) and anything whose min_ram_gb exceeds
    # free RAM — loading that would OOM-kill the kernel. The client re-checks
    # at load time too, so a model that slips through is skipped cleanly.
    import psutil
    _free = psutil.virtual_memory().available / 1e9
    _keep = []
    for m in MODELS:
        c = _mc.get(m, {})
        if c.get('load_in_4bit'):
            print(f'  drop {m}: 4-bit, GPU-only'); continue
        need = float(c.get('min_ram_gb', 0) or 0)
        if need and need > _free:
            print(f'  drop {m}: needs ~{need:.0f} GB, only {_free:.1f} GB free'); continue
        _keep.append(m)
    MODELS = _keep
    print(f'CPU mode ({_free:.1f} GB free) -> MODELS={MODELS}')

## 6. Run the sweep

Claims a batch, runs it, pushes the results, repeats until this runner
finds no unclaimed work left (or every pair hits `TARGET`, if you set one).
Run all in each Colab account. Safe to re-run after a timeout.

In [ ]:
import os, sys, shutil, subprocess, pathlib, random, time, json
for _p in (os.path.join(os.getcwd(), 'src'), os.getcwd()):
    if _p not in sys.path: sys.path.insert(0, _p)
from arabic_ocr_bench.runner import run as _run_batch, scored_indices, next_indices
from arabic_ocr_bench.report import score_run
from arabic_ocr_bench.coordination import GitLedger, runner_identity
from arabic_ocr_bench.samples import export_run_samples

CAPPED = True
DRIVE = pathlib.Path('/content/drive/MyDrive/arabic-ocr-bench')
DRIVE_OK = DRIVE.parent.exists()
if not DRIVE_OK:
    raise SystemExit('Multi-runner sweep needs the shared Google Drive mounted (cell 4) for the _cache / results mirror.')

# The lease ledger lives on a dedicated git branch, NOT on Drive: runners
# signed in to different Google accounts do not share a MyDrive tree, and a
# folder shared between accounts propagates writes far too slowly for
# leasing to work. git is the one store every runner sees consistently.
CLAIMS_REPO = pathlib.Path('/content/.sweep-claims-repo')
if not (CLAIMS_REPO / '.git').is_dir():
    _tok = os.environ.get('GH_TOKEN') or ''
    try:
        from google.colab import userdata
        _tok = _tok or (userdata.get('GH_TOKEN') or '')
    except Exception:
        pass
    _auth = f'{_tok}@' if _tok else ''
    _url = f'https://{_auth}github.com/rabeeh97/arabic-ocr-benchmark.git'
    subprocess.run(['git', 'clone', '--depth', '1', '--no-tags', _url, str(CLAIMS_REPO)],
                   check=True)
LEDGER = GitLedger(CLAIMS_REPO, branch='sweep-claims', ttl_sec=LEASE_TTL_MIN * 60)
CLAIMS = CLAIMS_REPO / 'claims'
RUNNER_ID, SESSION_ID = runner_identity()
print(f'this runner : {RUNNER_ID}')
print(f'session     : {SESSION_ID}')
print(f'claims (git): origin/sweep-claims  ->  {CLAIMS}')

_DS = {}
_HF_SPLIT = {
    'muharaf_lines': ('aamijar/muharaf-public', 'test'),
    'kitab_khatt': ('ahmedheakl/arocrbench_khatt', 'train'),
    'kitab_muharaf': ('ahmedheakl/arocrbench_muharaf', 'train'),
    'kitab_historicalbooks': ('ahmedheakl/arocrbench_historicalbooks', 'train'),
    'kitab_hindawi': ('ahmedheakl/arocrbench_hindawi', 'train'),
    'kitab_historyar': ('ahmedheakl/arocrbench_historyar', 'train'),
    'kitab_onlinekhatt': ('ahmedheakl/arocrbench_onlinekhatt', 'train'),
    'nakba_lines': ('U4RASD/omar-al-saleh-manuscripts-segments', 'test'),
}
def _get_image_fn(dataset):
    if dataset not in _HF_SPLIT:
        return lambda i: None
    if dataset not in _DS:
        from datasets import load_dataset
        hf_id, split = _HF_SPLIT[dataset]
        _DS[dataset] = load_dataset(hf_id, split=split)
    ds = _DS[dataset]
    return lambda i: ds[int(i)]['image']

def _git(*a):
    return subprocess.run(['git', *a], capture_output=True, text=True)

def _mirror_to_drive(run_dir):
    if not DRIVE_OK:
        return
    (DRIVE / 'results').mkdir(parents=True, exist_ok=True)
    if pathlib.Path('results/_cache').is_dir():
        shutil.copytree('results/_cache', DRIVE / '_cache', dirs_exist_ok=True)
    shutil.copytree(run_dir, DRIVE / 'results' / pathlib.Path(run_dir).name, dirs_exist_ok=True)

# size of each split, to know when a (model, dataset) pair is exhausted
try:
    _SIZES = json.load(open('results/_dataset_sizes.json'))
except Exception:
    _SIZES = {}
def _dsize(dataset):
    if dataset not in _SIZES:
        from arabic_ocr_bench.datasets.registry import build_dataset, load_dataset_configs
        _SIZES[dataset] = len(build_dataset(dataset, load_dataset_configs()[dataset]))
    return _SIZES[dataset]

def _persist(model, dataset, run_dir, n):
    """score -> samples -> Drive mirror -> git push (with retry). Raises only
    if the push cannot land after several rebase+retry rounds, so the batch is
    retried later (a 'done' lease always means the results are in the repo).
    The leaderboard is NOT built or pushed here — every runner regenerating and
    committing the same results/leaderboard.* every batch is the main source of
    push contention. Build it on demand from the last cell instead."""
    run_dir = str(run_dir)
    try:
        score_run(pathlib.Path(run_dir), profiles=('raw', 'light', 'diac_insensitive'), progress=False)
    except Exception as e:
        print('   scoring failed:', e)
    try:
        export_run_samples(run_dir, get_image=_get_image_fn(dataset), k=3)
    except Exception as e:
        print('   sample export failed:', e)
    # drop the decoded HF dataset we just loaded for the 3 sample images —
    # keeping every dataset resident in _DS is a slow system-RAM leak.
    _DS.clear(); import gc; gc.collect()
    _mirror_to_drive(run_dir)
    if not token:
        print('   no GH_TOKEN -> Drive only (still counts as persisted)')
        return
    _git('config', 'user.email', 'colab@arabic-ocr-bench')
    _git('config', 'user.name', f'colab:{RUNNER_ID}')
    _git('add', '-f', run_dir, 'results/_cache')
    _git('commit', '-m', f'sweep[{RUNNER_ID}]: {model} {dataset} n={n} ({pathlib.Path(run_dir).name})')
    last = ''
    for _try in range(8):
        _git('pull', '--rebase', '--autostash', 'origin', 'main')
        p = _git('push', 'origin', 'HEAD:main')
        if p.returncode == 0:
            print(f'   push: ok{"" if _try == 0 else f" (after {_try + 1} tries)"}')
            return
        last = (p.stderr or p.stdout).strip()[:200]
        time.sleep(random.uniform(2.0, 6.0))
    raise RuntimeError('git push failed after 8 tries: ' + (last or 'unknown'))

if DRIVE_OK and (DRIVE / '_cache').is_dir():
    shutil.copytree(DRIVE / '_cache', 'results/_cache', dirs_exist_ok=True)
    print('restored line cache from Drive')

RESULTS = pathlib.Path('results')

def _run_range(model, dataset, indices):
    cfg = {'models': [model], 'datasets': [dataset], 'indices': sorted(indices),
           'accumulate': True, 'cache': True, 'prompt_version': 'line-v2',
           'normalization_profiles': ['raw', 'light', 'diac_insensitive'],
           'concurrency': 1, 'results_dir': 'results',
           'runner_id': RUNNER_ID, 'session_id': SESSION_ID}
    rd = _run_batch(cfg)
    _persist(model, dataset, rd, len(indices))
    return rd

def _free_indices(model, dataset):
    done = scored_indices(RESULTS, model, dataset)
    held = LEDGER.covered_indices(model, dataset)
    if CAPPED and TARGET is not None:
        n = max(0, min(BATCH, TARGET - len(done)))
    else:
        n = BATCH
    if not n:
        return []
    cap = _dsize(dataset)
    return [i for i in next_indices(done | held, n) if i < cap]

def _do(model, dataset, indices, lease):
    try:
        rd = _run_range(model, dataset, indices)
    except Exception as e:
        import traceback; traceback.print_exc()
        print(f'!!! {model}/{dataset} failed ({type(e).__name__}: {e}) — lease left to expire / be reclaimed')
        import gc; gc.collect()
        return False
    LEDGER.complete(lease, pathlib.Path(rd).name, len(indices))
    import gc; gc.collect()   # reclaim between batches (RAM, not just VRAM)
    return True

# ---- 1. finish this runner's own interrupted batches first ----
for _lease in LEDGER.my_open(RUNNER_ID):
    print(f'resume own : {_lease.model}/{_lease.dataset} rows {_lease.i0}..{_lease.i1 - 1}')
    _do(_lease.model, _lease.dataset, _lease.indices, _lease)

# ---- 2. work loop ----
PAIRS = [(m, d) for d in DATASETS for m in MODELS]
random.shuffle(PAIRS)          # each runner starts from a different corner
exhausted = set()              # (model, dataset) with no rows left for this runner
failed = set()                 # (model, dataset) that errored — don't re-claim this session
idle = 0
while True:
    progressed = False

    # 2a. take over a crashed runner's abandoned batch, if any
    for _stale in LEDGER.reclaimable():
        _taken = LEDGER.adopt(_stale, RUNNER_ID, SESSION_ID)
        if _taken and LEDGER.confirm(_taken):
            print(f'reclaim    : {_taken.model}/{_taken.dataset} rows {_taken.i0}..{_taken.i1 - 1} (was {_stale.runner_id}, idle {_stale.age() / 60:.0f}m)')
            progressed = _do(_taken.model, _taken.dataset, _taken.indices, _taken)
        elif _taken:
            LEDGER.release(_taken)
        break   # one at a time, then re-read state
    if progressed:
        idle = 0; continue

    # 2b. fresh work
    for (_m, _d) in PAIRS:
        if (_m, _d) in exhausted or (_m, _d) in failed:
            continue
        _want = _free_indices(_m, _d)
        if not _want:
            exhausted.add((_m, _d)); continue
        _lease = LEDGER.claim(_m, _d, _want, RUNNER_ID, SESSION_ID)
        if _lease is None or not LEDGER.confirm(_lease):
            print(f'   lost race: {_m}/{_d} rows {_want[0]}..{_want[-1]} — trying elsewhere')
            if _lease is not None: LEDGER.release(_lease)
            continue
        print(f'{_m} / {_d}   rows {_want[0]}..{_want[-1]}  ({len(_want)})')
        progressed = _do(_m, _d, _want, _lease)
        if not progressed:
            failed.add((_m, _d))   # errored — leave the lease to expire, skip it here
        break        # re-read scored / leases / cache from the top

    if progressed:
        idle = 0
    else:
        idle += 1
        if idle >= 3:
            print('\nno unclaimed work left for this runner.')
            break
        time.sleep(15)

try:
    LEDGER.prune_done()
except Exception:
    pass
print(f'\n=== RUNNER DONE ===   audit:  python scripts/sweep_audit.py results --claims {CLAIMS}')

## 7. Final leaderboard

In [ ]:
!python scripts/make_leaderboard.py results -o results/leaderboard.html
from IPython.display import HTML
HTML(open('results/leaderboard.html').read())

## Notes — running several Colab accounts at once

This notebook is a **coordinated multi-runner sweep**. Point 2–3 Colab
accounts at the same repo, Run all in each, and they split the work between
them. The runners may be *different* Google accounts — they do not need to
share a Drive:

- **Claims ledger** lives on a dedicated git branch, `origin/sweep-claims`
  (a small throwaway clone at `/content/.sweep-claims-repo`), **not** on
  Drive — runners on different Google accounts don't share a `MyDrive` tree,
  and a shared folder syncs across accounts far too slowly to lease against.
  Before a batch a runner commits + pushes a lease file for an exact
  `(model, dataset, row-range)`; every runner fetches the branch first, so
  two never take the same range. GitHub serialises pushes to the ref, so a
  simultaneous claim becomes a rejected push: the loser rebases, re-applies
  the deterministic tiebreak (earliest timestamp, then session id), and
  moves on. Drive is still used, but only for the `_cache` / results mirror.
- **Runner identity** is your Colab Google-account email (falls back to a
  `/content/.sweep_runner_id` uuid), plus a fresh per-session id. Both are
  stamped onto every prediction row and every lease, so you can always tell
  who computed what.
- **Recovery.** A `reserved` lease older than `LEASE_TTL_MIN` is treated as
  abandoned and any runner reclaims it. On start a runner also re-runs its
  *own* still-open leases first (a crash mid-batch, before the results were
  saved).
- **Nothing is overwritten.** Each runner writes its own `results/<run_id>/`
  dirs and pushes them; the per-line cache (shared via Drive + git) stops
  genuine duplicate spend. Any accidental double-work is visible in
  `python scripts/sweep_audit.py results --claims /content/.sweep-claims-repo/claims`.
- A single runner works fine too — it just claims and clears every batch
  itself.
- **Force a re-run** of one model: delete its `results/<run_id>/` dirs and its
  `results/_cache/` entries so those lines look un-scored again.
- Hosted models (Gemini / GPT / Claude): `02_colab_hosted_apis.ipynb`.